# ViT tier: frozen ViTs as global descriptors

Reads `results/runs.jsonl` and plots it; producing rows is `cbir evaluate`'s job. The
companion to `03_cnn_comparison.ipynb`, and `06_vit_training.ipynb` covers the
trainable half.

Every row here is a **frozen** network read out by its own image embedding — CLS after
the final LayerNorm for DINOv2, DINOv3 and DeiT, the attention-pooling head for SigLIP 2.
The headline protocol is DINOv3 App. D.8: longer side to 224, centred on a black square
(`pad`), bbx-cropped queries, cosine, nothing fitted. Anything else is an ablation and is
labelled as one.

Colour is the **pretraining objective** (self-supervised, image-text, supervised), not
the model family — the objective is the finding. All mAP on 0-1.

```bash
uv run cbir evaluate --dataset roxford5k vit --model dinov3-vitl16
uv run cbir evaluate --dataset roxford5k vit --model siglip2-vitl16-512 --image-size 512 --whiten
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, ".")  # style.py sits beside this notebook
import style
from cbir.eval.frame import tidy_rows
from cbir.eval.results import latest

style.use()
FIGURES = Path("..") / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

## The data

One row per (run x protocol). The earliest ViT rows predate some knobs, so a
missing key takes the value the code had then — which is also today's default. This table
is the accessible view of every figure below.

In [ ]:
df = pd.DataFrame(tidy_rows(records=list(latest().values())))
vit = df[df["technique"] == "vit"].copy()

DEFAULTS = {"resize": "pad", "readout": "native", "interpolation": "bicubic",
            "whiten": False, "whiten_source": None, "checkpoint": None}
for key, default in DEFAULTS.items():
    vit[key] = [p.get(key, default) for p in vit["params"]]
vit["model"] = [p["model"] for p in vit["params"]]
vit["image_size"] = [p["image_size"] for p in vit["params"]]
# `whiten_source` is recorded even when nothing is whitened; it only means something with it.
vit.loc[~vit["whiten"].astype(bool), "whiten_source"] = None

FAMILY = {"dinov2": "self-supervised", "dinov3": "self-supervised", "siglip2": "image-text", "deit": "supervised"}
vit["objective"] = vit["model"].str.split("-").str[0].map(FAMILY)

NAME = {
    "dinov2-vits14": "DINOv2 S/14", "dinov2-vitb14": "DINOv2 B/14", "dinov2-vitl14": "DINOv2 L/14",
    "dinov2-vitg14": "DINOv2 g/14", "dinov3-vits16": "DINOv3 S/16", "dinov3-vitb16": "DINOv3 B/16",
    "dinov3-vitl16": "DINOv3 L/16", "siglip2-vitl16-256": "SigLIP 2 L/16 256",
    "siglip2-vitl16-384": "SigLIP 2 L/16 384", "siglip2-vitl16-512": "SigLIP 2 L/16 512",
    "deit-vits16-dist-224": "DeiT S/16† 224", "deit-vitb16-dist-224": "DeiT B/16† 224",
    "deit-vitb16-dist-384": "DeiT B/16† 384",
}
vit["name"] = vit["model"].map(NAME)

frozen = vit[vit["checkpoint"].isna()]
print(f"{len(frozen) // 3} frozen run(s) across {frozen['model'].nunique()} models")
frozen[["dataset", "name", "image_size", "resize", "readout", "whiten_source", "protocol", "map"]].head(12)

## Every model at the protocol

Each model at its protocol size — 224 for DINO, the native size for the fixed-resolution
SigLIP 2 and DeiT checkpoints — padded, CLS or pooling head, no whitening. On roxford5k
the best classic row (VLAD) and the fine-tuned CNN (Radenović et al.'s GeM ResNet101) are
drawn as rules: the comparison this tier exists for. rparis6k has no rules, because the
earlier tiers were measured on roxford5k only.

The tick on DINOv3 L/16 Hard is the DINOv3 paper's published 63.1 — the ViT
tier's validation row.

In [ ]:
def at_protocol(rows):
    native = rows["model"].str.startswith(("siglip2", "deit"))
    return rows[(rows["resize"] == "pad") & (rows["readout"] == "native") & ~rows["whiten"].astype(bool)
                & (rows["interpolation"] == "bicubic") & (native | (rows["image_size"] == 224))]

protocol = at_protocol(frozen)
order = (protocol[(protocol["dataset"] == "roxford5k") & (protocol["protocol"] == "medium")]
         .sort_values("map")["name"].tolist())

# The other tiers' best rows on roxford5k, from the same record.
def best(technique, protocol_name, finetuned=None):
    rows = df[(df["technique"] == technique) & (df["dataset"] == "roxford5k") & (df["protocol"] == protocol_name)]
    if finetuned is not None:
        rows = rows[[(p.get("weights") == "sfm120k") == finetuned and not p.get("checkpoint") for p in rows["params"]]]
    return rows["map"].max()

RULES = {p: [("VLAD (classic)", best("vlad", p)), ("GeM R101 fine-tuned", best("gem", p, finetuned=True))]
         for p in ("medium", "hard")}
PUBLISHED = {("roxford5k", "hard", "DINOv3 L/16"): 0.631}

fig_models, axes = plt.subplots(2, 2, figsize=(11, 8.2), sharey=True)
for row, dataset in enumerate(["roxford5k", "rparis6k"]):
    for col, protocol_name in enumerate(["medium", "hard"]):
        ax = axes[row, col]
        rows = protocol[(protocol["dataset"] == dataset) & (protocol["protocol"] == protocol_name)].set_index("name")
        names = [n for n in order if n in rows.index]
        values = [rows.loc[n, "map"] for n in names]
        colours = [style.OBJECTIVE[rows.loc[n, "objective"]] for n in names]
        ax.barh(range(len(names)), values, 0.7, color=colours, zorder=3)
        for i, value in enumerate(values):
            style.label_point(ax, value, i, f"{value:.3f}", dx=4)
        if dataset == "roxford5k":
            for label, value in RULES[protocol_name]:
                ax.axvline(value, color=style.MUTED, linewidth=1.1, linestyle=(0, (4, 3)), zorder=2)
                # At the foot of the rule, clear of the panel title.
                style.label_point(ax, value, -0.6, label, dx=3, va="bottom", rotation=90, fontsize=7.5)
        for (d, p, name), value in PUBLISHED.items():
            if d == dataset and p == protocol_name and name in names:
                i = names.index(name)
                ax.plot([value, value], [i - 0.42, i + 0.42], color=style.INK, linewidth=2, zorder=4)
                style.label_point(ax, value, i + 0.42, f"published {value:.3f}", dx=0, dy=4, ha="center", va="bottom", fontsize=7)
        ax.set_yticks(range(len(names)), names)
        ax.set_xlim(0, 1.0)
        style.style(ax, title=f"{dataset} {protocol_name.capitalize()}", xlabel="mAP" if row else None, axis="x")

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in style.OBJECTIVE.values()]
fig_models.legend(handles, list(style.OBJECTIVE), loc="upper center", ncol=3, frameon=False,
                  bbox_to_anchor=(0.5, 1.0), title="pretraining objective")
fig_models.tight_layout(rect=(0, 0, 1, 0.93))  # the top band is the legend's

Three things to read off it:

- **The objective decides the tier, not the architecture.** All thirteen are ViTs. On
  roxford5k Medium the supervised DeiT classifiers sit below VLAD, the image-text SigLIP 2
  level with it, and every self-supervised DINO above both. Four of the seven DINO models
  (DINOv3 L and B, DINOv2 L and g) clear the fine-tuned CNN too — DINOv3 L/16 by 0.156
  Medium and 0.233 Hard, frozen, against a CNN trained for this exact task.
- **The validation row reproduces:** 0.637 Hard against the DINOv3 paper's 63.1.
- **Scale helps within a family until it doesn't:** DINOv2 g/14 lands *below* L/14 on
  roxford5k, as in the DINOv2 paper's own table.

## Resolution

224 is the protocol; larger inputs are an ablation. Only the DINO models can take them —
SigLIP 2 and DeiT have one checkpoint per size. The y-axis does not start at zero here:
the subject is the change between sizes, not the magnitude.

In [ ]:
res = frozen[frozen["model"].str.startswith(("dinov2", "dinov3")) & (frozen["resize"] == "pad")
             & (frozen["readout"] == "native") & ~frozen["whiten"].astype(bool)
             & (frozen["interpolation"] == "bicubic") & (frozen["dataset"] == "roxford5k")]
sized = res.groupby("model")["image_size"].nunique()
res = res[res["model"].isin(sized[sized > 1].index)]

MARKERS = ["o", "s", "^", "D", "v", "P"]

def dodge(ends, gap):
    # Spread end-of-line labels apart vertically so close lines stay readable.
    placed = []
    for y, text in sorted(ends):
        y = max(y, placed[-1][0] + gap) if placed else y
        placed.append((y, text))
    return placed

fig_res, axes = plt.subplots(1, 2, figsize=(10.5, 3.9))
for ax, protocol_name in zip(axes, ["medium", "hard"]):
    ends = []
    for marker, (model, rows) in zip(MARKERS, res[res["protocol"] == protocol_name].groupby("model")):
        rows = rows.sort_values("image_size")
        ax.plot(rows["image_size"], rows["map"], color=style.OBJECTIVE["self-supervised"], linewidth=2,
                marker=marker, markersize=7, markeredgecolor=style.SURFACE, markeredgewidth=1.5, zorder=3)
        ends.append((rows["map"].iloc[-1], NAME[model]))
    low, high = res[res["protocol"] == protocol_name]["map"].agg(["min", "max"])
    for y, text in dodge(ends, gap=(high - low) * 0.07):
        ax.annotate(text, (520, y), xytext=(10, 0), textcoords="offset points", fontsize=8, color=style.MUTED, va="center")
    # DINOv2's largest size is 518 (37 patches of 14), DINOv3's 512 (32 of 16): one tick.
    ax.set_xticks([224, 448, 515], ["224", "448", "512 / 518"])
    ax.set_xlim(200, 640)
    style.style(ax, title=f"roxford5k {protocol_name.capitalize()}", xlabel="input side (px)",
                ylabel="mAP" if protocol_name == "medium" else None)
fig_res.tight_layout()

Resolution is not a free gain. It lifts DINOv2 S, B and g and *costs* DINOv2 L; DINOv3
L/16 is flat across 224-512. None of it changes which model leads, so 224 stays the
protocol and the larger sizes stay an ablation.

## Post-processing and readout

Two ablations on top of the protocol rows, both measured rather than assumed:

- **Whitening.** PCA-whitening is what the CNN tier's off-the-shelf rows need. Here it
  depends on the objective: it *costs* DINO and *helps* SigLIP 2.
- **Readout.** CLS (or SigLIP's pooling head) against mean and GeM (`p = 3`) over the
  patch tokens. Pooled readouts are measured unpadded (`aspect`), because under `pad` the
  black bars are patches and get averaged in. DeiT and SigLIP 2 cannot run unpadded, so
  their pooled bars are confounded and drawn hatched.

In [ ]:
SUBJECTS = ["dinov3-vitl16", "dinov2-vitl14", "siglip2-vitl16-512", "deit-vitb16-dist-384"]
rox = frozen[(frozen["dataset"] == "roxford5k") & (frozen["protocol"] == "medium")
             & frozen["model"].isin(SUBJECTS) & (frozen["interpolation"] == "bicubic")]
native_size = rox["model"].str.startswith(("siglip2", "deit")) | (rox["image_size"] == 224)
rox = rox[native_size]

def value(model, readout="native", resize="pad", source=None):
    rows = rox[(rox["model"] == model) & (rox["readout"] == readout) & (rox["resize"] == resize)]
    rows = rows[rows["whiten_source"].isna()] if source is None else rows[rows["whiten_source"] == source]
    return rows["map"].max() if len(rows) else float("nan")

def pooled(model, readout):
    # Unpadded where the model allows it; padded (and so confounded) where it does not.
    unpadded = value(model, readout, "aspect")
    return (unpadded, False) if unpadded == unpadded else (value(model, readout, "pad"), True)

PANELS = {
    "Whitening (roxford5k Medium)": [
        ("none", lambda m: (value(m), False)),
        ("PCA-w held-out", lambda m: (value(m, source="held_out"), False)),
        ("PCA-w sfm30k", lambda m: (value(m, source="sfm30k"), False)),
    ],
    "Readout (roxford5k Medium)": [
        ("CLS / head", lambda m: (value(m), False)),
        ("mean tokens", lambda m: pooled(m, "mean")),
        ("GeM tokens", lambda m: pooled(m, "gem")),
    ],
}

fig_ablate, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True)
width = 0.26
for ax, (title, conditions) in zip(axes, PANELS.items()):
    for j, (label, read) in enumerate(conditions):
        for i, model in enumerate(SUBJECTS):
            v, confounded = read(model)
            if v != v:
                continue
            objective = FAMILY[model.split("-")[0]]
            x = i + (j - 1) * width
            ax.bar(x, v, width * 0.9, color=style.OBJECTIVE[objective], alpha=1.0 - 0.3 * j,
                   hatch="///" if confounded else None, edgecolor=style.SURFACE, linewidth=1.2, zorder=3)
            style.label_point(ax, x, v, f"{v:.2f}", dx=0, dy=4, ha="center", va="bottom", fontsize=7)
            style.label_point(ax, x, 0, label, dx=0, dy=4, ha="center", va="bottom", rotation=90, fontsize=7, color=style.SURFACE)
    ax.set_xticks(range(len(SUBJECTS)), [NAME[m] for m in SUBJECTS], fontsize=8.5)
    ax.set_ylim(0, 0.95)
    style.style(ax, title=title, ylabel="mAP" if ax is axes[0] else None)
fig_ablate.tight_layout()

- **Do not whiten DINO.** It costs DINOv3 L/16 0.020 Medium fitted on the held-out set
  and 0.032 on sfm30k — more fitting data, bigger loss, so it is not estimation noise.
  SigLIP 2 gains from it, which is why the cross-tier table compares each tier at its
  own best post-processing.
- **CLS beats pooled patch tokens by about a third of the scale on DINO**, even with the
  pad confound removed. The pooled readouts that serve a CNN's conv map are the wrong
  readout for a ViT trained to put the image in its CLS token.

## Write the docs figures

In [ ]:
# White rather than transparent: GitHub renders READMEs on a dark background too, and a
# transparent PNG would leave the dark ink invisible there.
for name, figure in (("vit_models", fig_models), ("vit_resolution", fig_res),
                     ("vit_ablations", fig_ablate)):
    path = FIGURES / f"{name}.png"
    figure.savefig(path, dpi=110, bbox_inches="tight", facecolor="white")
    print("wrote", path.resolve())